# 模型加载、部署、量化、剪枝、微调

### 检查所需的库 & 尽量在wsl里学习

In [1]:
pip show torch transformers accelerate

Name: torch
Version: 2.8.0+cu126
Summary: Tensors and Dynamic neural networks in Python with strong GPU acceleration
Home-page: https://pytorch.org/
Author: PyTorch Team
Author-email: packages@pytorch.org
License: BSD-3-Clause
Location: /home/kokomi/anaconda3/envs/mamba/lib/python3.12/site-packages
Requires: filelock, fsspec, jinja2, networkx, nvidia-cublas-cu12, nvidia-cuda-cupti-cu12, nvidia-cuda-nvrtc-cu12, nvidia-cuda-runtime-cu12, nvidia-cudnn-cu12, nvidia-cufft-cu12, nvidia-cufile-cu12, nvidia-curand-cu12, nvidia-cusolver-cu12, nvidia-cusparse-cu12, nvidia-cusparselt-cu12, nvidia-nccl-cu12, nvidia-nvjitlink-cu12, nvidia-nvtx-cu12, setuptools, sympy, triton, typing-extensions
Required-by: accelerate, bitsandbytes, compressed-tensors, flash_attn, flashinfer-python, peft, quack-kernels, tilelang, tokenspeed-mla, torch_c_dlpack_ext, torchaudio, torchvision, vllm, xformers, xgrammar
---
Name: transformers
Version: 5.5.4
Summary: Transformers: the model-definition framework for state-o

In [2]:
#使用modelscope下载模型
# !pip install modelscope
!pip show modelscope

Name: modelscope
Version: 1.37.1
Summary: ModelScope: bring the notion of Model-as-a-Service to life.
Home-page: https://github.com/modelscope/modelscope
Author: ModelScope team
Author-email: contact@modelscope.cn
License-Expression: Apache-2.0
Location: /home/kokomi/anaconda3/envs/mamba/lib/python3.12/site-packages
Requires: filelock, packaging, requests, setuptools, tqdm, urllib3
Required-by: 


In [3]:
# !pip install vllm==0.10.2 openai
!pip show vllm openai

Name: vllm
Version: 0.10.2
Summary: A high-throughput and memory-efficient inference and serving engine for LLMs
Home-page: https://github.com/vllm-project/vllm
Author: vLLM Team
Author-email: 
License-Expression: Apache-2.0
Location: /home/kokomi/anaconda3/envs/mamba/lib/python3.12/site-packages
Requires: aiohttp, blake3, cachetools, cbor2, cloudpickle, compressed-tensors, depyf, diskcache, einops, fastapi, filelock, gguf, lark, llguidance, lm-format-enforcer, mistral_common, msgspec, ninja, numba, numpy, openai, openai-harmony, opencv-python-headless, outlines_core, partial-json-parser, pillow, prometheus-fastapi-instrumentator, prometheus_client, protobuf, psutil, py-cpuinfo, pybase64, pydantic, python-json-logger, pyyaml, pyzmq, ray, regex, requests, scipy, sentencepiece, setproctitle, setuptools, six, tiktoken, tokenizers, torch, torchaudio, torchvision, tqdm, transformers, typing_extensions, watchfiles, xformers, xgrammar
Required-by: 
---
Name: openai
Version: 2.37.0
Summary: Th

In [4]:
# pip install datasets peft trl
!pip show datasets peft trl

Name: datasets
Version: 4.8.5
Summary: HuggingFace community-driven open-source library of datasets
Home-page: https://github.com/huggingface/datasets
Author: HuggingFace Inc.
Author-email: thomas@huggingface.co
License: Apache 2.0
Location: /home/kokomi/anaconda3/envs/mamba/lib/python3.12/site-packages
Requires: dill, filelock, fsspec, httpx, huggingface-hub, multiprocess, numpy, packaging, pandas, pyarrow, pyyaml, requests, tqdm, xxhash
Required-by: trl
---
Name: peft
Version: 0.19.1
Summary: Parameter-Efficient Fine-Tuning (PEFT)
Home-page: https://github.com/huggingface/peft
Author: The HuggingFace team
Author-email: benjamin@huggingface.co
License: Apache
Location: /home/kokomi/anaconda3/envs/mamba/lib/python3.12/site-packages
Requires: accelerate, huggingface_hub, numpy, packaging, psutil, pyyaml, safetensors, torch, tqdm, transformers
Required-by: 
---
Name: trl
Version: 1.4.0
Summary: Train transformer language models with reinforcement learning.
Home-page: https://github.com/h

In [5]:
import gc
import torch
from modelscope import AutoModelForCausalLM, AutoTokenizer
from transformers import BitsAndBytesConfig, TrainingArguments, Trainer, DataCollatorForSeq2Seq

### GPU用量检测和缓存清理

In [6]:
def print_gpu_memory(tag=""):
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**3
        reserved = torch.cuda.memory_reserved() / 1024**3
        total = torch.cuda.get_device_properties(0).total_memory / 1024**3
        free = total - allocated
        print(f"[{tag}] 已用: {allocated:.2f}GB | 缓存: {reserved:.2f}GB | 总计: {total:.2f}GB | 剩余: {free:.2f}GB")
    else:
        print("CUDA不可用")

In [7]:
def CleanMemory():
    torch.cuda.empty_cache()
    gc.collect()
    print_gpu_memory("缓存已清理")

## Qwen/Qwen3-0.6B纯文本模型加载

**下载模型**

In [13]:
!modelscope download --model Qwen/Qwen3-0.6B --local_dir ./Qwen/Qwen3-0.6B


 _   .-')                _ .-') _     ('-.             .-')                              _ (`-.    ('-.
( '.( OO )_             ( (  OO) )  _(  OO)           ( OO ).                           ( (OO  ) _(  OO)
 ,--.   ,--.).-'),-----. \     .'_ (,------.,--.     (_)---\_)   .-----.  .-'),-----.  _.`     \(,------.
 |   `.'   |( OO'  .-.  ',`'--..._) |  .---'|  |.-') /    _ |   '  .--./ ( OO'  .-.  '(__...--'' |  .---'
 |         |/   |  | |  ||  |  \  ' |  |    |  | OO )\  :` `.   |  |('-. /   |  | |  | |  /  | | |  |
 |  |'.'|  |\_) |  |\|  ||  |   ' |(|  '--. |  |`-' | '..`''.) /_) |OO  )\_) |  |\|  | |  |_.' |(|  '--.
 |  |   |  |  \ |  | |  ||  |   / : |  .--'(|  '---.'.-._)   \ ||  |`-'|   \ |  | |  | |  .___.' |  .--'
 |  |   |  |   `'  '-'  '|  '--'  / |  `---.|      | \       /(_'  '--'\    `'  '-'  ' |  |      |  `---.
 `--'   `--'     `-----' `-------'  `------'`------'  `-----'    `-----'      `-----'  `--'      `------'


Successfully Downloaded from model Qwen/Qwen3-0.6B.


** 加载Qwen/Qwen3-0.6B**

In [8]:
def load_model(model_name):
    # 加载分词器
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    # 加载模型
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        dtype=torch.bfloat16,
        # dtype=torch.float16, #上面的不能用就用下面的
        device_map="auto"
    )
    return model, tokenizer

**准备prompt并推理得到输出**

In [9]:
# 模型存放地址或模型名
model_name = "./Qwen/Qwen3-0.6B"
# 准备 prompt
prompt = "什么是VLA，有什么用"
# 封装到 messages
messages = [
    {"role": "user", "content": prompt}
]
# 加载模型
print_gpu_memory("加载模型前")
model, tokenizer = load_model(model_name)
print_gpu_memory("加载模型后")

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=True # 是否使用思考模式
)
# 编码
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
# 推理
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=32768 #最大上下文长度
)
# 得到输出并转 ids
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 
try:
    # 找到思考结束的符号的id的位置 151668 (</think>)
    index = len(output_ids) - output_ids[::-1].index(151668)
except ValueError:
    index = 0
# 思考内容
thinking_content = tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip("\n")
# 输出内容
content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")

print("thinking content:", thinking_content)
print("content:", content)

[加载模型前] 已用: 0.00GB | 缓存: 0.00GB | 总计: 6.00GB | 剩余: 6.00GB


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

[加载模型后] 已用: 1.11GB | 缓存: 1.55GB | 总计: 6.00GB | 剩余: 4.89GB
thinking content: <think>
嗯，用户问的是VLA是什么，以及它的用途。首先，我需要确认VLA的全称和常见含义。可能用户指的是“Variable Length Array”（变长数组）或者“Variable Length Encoding”（变长编码），但根据常见的用法，VLA应该指的是变量长度数组。

接下来，用户可能对VLA的概念不太清楚，需要解释清楚。可能需要先说明VLA的基本定义，然后说明它的用途。比如，VLA在计算机科学中常用于处理动态内存分配，尤其是在C或C++语言中。用户可能想知道为什么需要动态分配，以及VLA在实际应用中的例子。

另外，用户可能对VLA和静态数组的区别感兴趣，或者想知道VLA在不同编程语言中的实现差异。需要确保回答准确，避免混淆。可能还需要提到VLA在内存管理中的作用，比如优化内存使用效率，减少内存泄漏等问题。

最后，要检查是否有其他可能的解释，比如VLA是否指其他技术，但根据常见知识，应该是正确的。总结时要简明扼要，帮助用户理解VLA的基本概念和实际应用。
</think>
content: **VLA（Variable Length Array，变长数组）** 是计算机科学中用于动态内存分配的一种技术，其核心思想是**根据需要动态调整数组的大小**，而非静态分配固定长度的数组。

### **VLA的用途：**
1. **动态内存管理**：  
   - 在C/C++等语言中，VLA允许在运行时分配和释放内存，无需预先定义数组的大小，适用于需要灵活处理内存的场景。

2. **优化内存使用**：  
   - 通过动态调整数组大小，可以避免因静态分配导致的内存浪费，特别是在处理动态数据结构（如列表、链表等）时。

3. **处理复杂数据结构**：  
   - 适用于需要灵活扩展或缩小数据集的程序，例如实时数据处理、图形渲染等场景。

### **与静态数组的区别**：  
- **静态数组**：固定长度，内存分配和释放固定，适用于简单数据结构。  
- **VLA**：根据需要动态调整大小，更灵活，但可能在某些语言或环境中实现较为复杂。

### **实际例子**：  
- 在C语言中，V

In [10]:
# 清理缓存
del generated_ids
del model_inputs
del model
del tokenizer
CleanMemory()

[缓存已清理] 已用: 0.01GB | 缓存: 0.15GB | 总计: 6.00GB | 剩余: 5.99GB


## Qwen/Qwen3-0.6B纯文本模型本地部署

**安装vllm和openai库**

In [11]:
# !pip install vllm==0.10.2 openai
!pip show vllm openai

Name: vllm
Version: 0.10.2
Summary: A high-throughput and memory-efficient inference and serving engine for LLMs
Home-page: https://github.com/vllm-project/vllm
Author: vLLM Team
Author-email: 
License-Expression: Apache-2.0
Location: /home/kokomi/anaconda3/envs/mamba/lib/python3.12/site-packages
Requires: aiohttp, blake3, cachetools, cbor2, cloudpickle, compressed-tensors, depyf, diskcache, einops, fastapi, filelock, gguf, lark, llguidance, lm-format-enforcer, mistral_common, msgspec, ninja, numba, numpy, openai, openai-harmony, opencv-python-headless, outlines_core, partial-json-parser, pillow, prometheus-fastapi-instrumentator, prometheus_client, protobuf, psutil, py-cpuinfo, pybase64, pydantic, python-json-logger, pyyaml, pyzmq, ray, regex, requests, scipy, sentencepiece, setproctitle, setuptools, six, tiktoken, tokenizers, torch, torchaudio, torchvision, tqdm, transformers, typing_extensions, watchfiles, xformers, xgrammar
Required-by: 
---
Name: openai
Version: 2.37.0
Summary: Th

### 部署在本地的8000端口

**旧版写法**

In [2]:
!python -m vllm.entrypoints.openai.api_server --model ./Qwen/Qwen3-0.6B --host 127.0.0.1 --port 8000 --gpu-memory-utilization 0.5

INFO 05-22 13:24:42 [__init__.py:216] Automatically detected platform cuda.
(APIServer pid=12142) INFO 05-22 13:24:43 [api_server.py:1896] vLLM API server version 0.10.2
(APIServer pid=12142) INFO 05-22 13:24:43 [utils.py:328] non-default args: {'host': '127.0.0.1', 'model': './Qwen/Qwen3-0.6B', 'gpu_memory_utilization': 0.5}
(APIServer pid=12142) INFO 05-22 13:24:47 [__init__.py:742] Resolved architecture: Qwen3ForCausalLM
(APIServer pid=12142) `torch_dtype` is deprecated! Use `dtype` instead!
(APIServer pid=12142) INFO 05-22 13:24:47 [__init__.py:1815] Using max model len 40960
(APIServer pid=12142) INFO 05-22 13:24:48 [scheduler.py:222] Chunked prefill is enabled with max_num_batched_tokens=2048.
INFO 05-22 13:24:50 [__init__.py:216] Automatically detected platform cuda.
(EngineCore_DP0 pid=12352) INFO 05-22 13:24:51 [core.py:654] Waiting for init message from front-end.
(EngineCore_DP0 pid=12352) INFO 05-22 13:24:51 [core.py:76] Initializing a V1 LLM engine (v0.10.2) with config: m

**新版写法**

In [ ]:
!vllm serve ./Qwen/Qwen3-0.6B --host 127.0.0.1 --port 8000 --gpu-memory-utilization 0.5

**ipynb里为了不阻塞后续代码可以这样启动**

In [22]:
import subprocess

cmd = "vllm serve ./Qwen/Qwen3-0.6B --host 127.0.0.1 --port 8000 --gpu-memory-utilization 0.5"
# 启动vLLM服务器（记录进程对象）
process = subprocess.Popen(
    cmd.split(),
    # stdout=subprocess.DEVNULL,   # 丢弃所有正常输出（日志）
    # stderr=subprocess.DEVNULL    # 丢弃所有错误输出
)

INFO 05-22 13:54:19 [__init__.py:216] Automatically detected platform cuda.
(APIServer pid=15568) INFO 05-22 13:54:21 [api_server.py:1896] vLLM API server version 0.10.2
(APIServer pid=15568) INFO 05-22 13:54:21 [utils.py:328] non-default args: {'model_tag': './Qwen/Qwen3-0.6B', 'host': '127.0.0.1', 'model': './Qwen/Qwen3-0.6B', 'gpu_memory_utilization': 0.5}
(APIServer pid=15568) INFO 05-22 13:54:25 [__init__.py:742] Resolved architecture: Qwen3ForCausalLM
(APIServer pid=15568) INFO 05-22 13:54:25 [__init__.py:1815] Using max model len 40960


(APIServer pid=15568) `torch_dtype` is deprecated! Use `dtype` instead!


(APIServer pid=15568) INFO 05-22 13:54:26 [scheduler.py:222] Chunked prefill is enabled with max_num_batched_tokens=2048.
INFO 05-22 13:54:29 [__init__.py:216] Automatically detected platform cuda.
(EngineCore_DP0 pid=15775) INFO 05-22 13:54:30 [core.py:654] Waiting for init message from front-end.
(EngineCore_DP0 pid=15775) INFO 05-22 13:54:30 [core.py:76] Initializing a V1 LLM engine (v0.10.2) with config: model='./Qwen/Qwen3-0.6B', speculative_config=None, tokenizer='./Qwen/Qwen3-0.6B', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=40960, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto, device_config=cuda, decoding_config=DecodingConfig(backend='auto', disable_fallback=False, disable_any_whitespace=False, disable_additional_properti

[W522 13:54:32.218020838 ProcessGroupNCCL.cpp:981] Warning: TORCH_NCCL_AVOID_RECORD_STREAMS is the default now, this environment variable is thus deprecated. (function operator())


[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
(EngineCore_DP0 pid=15775) INFO 05-22 13:54:32 [parallel_state.py:1165] rank 0 in world size 1 is assigned as DP rank 0, PP rank 0, TP rank 0, EP rank 0
(EngineCore_DP0 pid=15775) INFO 05-22 13:54:32 [topk_topp_sampler.py:58] Using FlashInfer for top-p & top-k sampling.
(EngineCore_DP0 pid=15775) INFO 05-22 13:54:32 [gpu_model_runner.py:2338] Starting to load model ./Qwen/Qwen3-0.6B...
(EngineCore_DP0 pid=15775) INFO 05-22 13:54:32 [gpu_model_runner.

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:00<00:00,  1.13it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:00<00:00,  1.13it/s]
(EngineCore_DP0 pid=15775) 


(EngineCore_DP0 pid=15775) INFO 05-22 13:54:33 [default_loader.py:268] Loading weights took 0.90 seconds
(EngineCore_DP0 pid=15775) INFO 05-22 13:54:34 [gpu_model_runner.py:2392] Model loading took 1.1201 GiB and 1.048009 seconds
(EngineCore_DP0 pid=15775) INFO 05-22 13:54:37 [backends.py:539] Using cache directory: /home/kokomi/.cache/vllm/torch_compile_cache/440b14807d/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=15775) INFO 05-22 13:54:37 [backends.py:550] Dynamo bytecode transform time: 2.94 s
(EngineCore_DP0 pid=15775) INFO 05-22 13:54:38 [backends.py:161] Directly load the compiled graph(s) for dynamic shape from the cache, took 1.392 s
(EngineCore_DP0 pid=15775) INFO 05-22 13:54:39 [monitor.py:34] torch.compile takes 2.94 s in total
(EngineCore_DP0 pid=15775) INFO 05-22 13:54:39 [gpu_worker.py:298] Available KV cache memory: 6.31 GiB
(EngineCore_DP0 pid=15775) INFO 05-22 13:55:15 [kv_cache_utils.py:864] GPU KV cache size: 59,040 tokens
(EngineCore_DP0 pid=15775

(EngineCore_DP0 pid=15775) 2026-05-22 13:55:15,155 - INFO - autotuner.py:457 - flashinfer.jit: [Autotuner]: Autotuning process starts ...
(EngineCore_DP0 pid=15775) 2026-05-22 13:55:15,203 - INFO - autotuner.py:466 - flashinfer.jit: [Autotuner]: Autotuning process ends
Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 67/67 [00:01<00:00, 50.86it/s]


(EngineCore_DP0 pid=15775) INFO 05-22 13:54:41 [gpu_model_runner.py:3118] Graph capturing finished in 2 secs, took 0.23 GiB
(EngineCore_DP0 pid=15775) INFO 05-22 13:54:41 [gpu_worker.py:391] Free memory on device (14.55/15.92 GiB) on startup. Desired GPU memory utilization is (0.5, 7.96 GiB). Actual usage is 1.12 GiB for weight, 0.52 GiB for peak activation, 0.01 GiB for non-torch memory, and 0.23 GiB for CUDAGraph memory. Replace gpu_memory_utilization config with `--kv-cache-memory=6365746688` to fit into requested memory, or `--kv-cache-memory=13443339776` to fully utilize gpu memory. Current kv cache memory in use is 6772594176 bytes.
(EngineCore_DP0 pid=15775) INFO 05-22 13:54:41 [core.py:218] init engine (profile, create kv cache, warmup model) took 7.53 seconds
(APIServer pid=15568) INFO 05-22 13:54:42 [loggers.py:142] Engine 000: vllm cache_config_info with initialization after num_gpu_blocks is: 3690
(APIServer pid=15568) INFO 05-22 13:54:42 [async_llm.py:180] Torch profiler d

(APIServer pid=15568) INFO:     Started server process [15568]
(APIServer pid=15568) INFO:     Waiting for application startup.
(APIServer pid=15568) INFO:     Application startup complete.


### 使用OpenAI标准接口调用

In [20]:
from openai import OpenAI

client = OpenAI(
    base_url="http://127.0.0.1:8000/v1",  # 服务提供商的地址，本地qwen系列这样写
    api_key="anything"                    # 本地模型的apikey可以随便填
)

response = client.chat.completions.create(
    model="./Qwen/Qwen3-0.6B",
    messages=[
        {"role": "user", "content": "一段话介绍一下苏州大学 /no_think"}  # /no_think表示不思考， /think表示要思考
    ],
    max_tokens=8196,
    stream=False  # 流式对话，逐字返回而不是等全部生成完再返回
)

print(response.choices[0].message.content)

(APIServer pid=14743) INFO 05-22 13:45:54 [chat_utils.py:538] Detected the chat template content format to be 'string'. You can set `--chat-template-content-format` to override this.
(APIServer pid=14743) INFO:     127.0.0.1:58668 - "POST /v1/chat/completions HTTP/1.1" 200 OK
<think>

</think>

苏州大学是一所位于中国江苏省苏州市的综合性大学，是江苏省的重点高校之一，也是中国“双一流”建设高校之一。学校致力于培养高水平的科学技术人才，注重学术研究与实践相结合，拥有较强的师资力量和丰富的学术资源。苏州大学在多个学科领域取得了显著成就，拥有多个国家级重点实验室和研究中心，为地方经济发展和社会进步做出了积极贡献。
(APIServer pid=14743) INFO 05-22 13:46:39 [loggers.py:123] Engine 000: Avg prompt throughput: 1.7 tokens/s, Avg generation throughput: 8.4 tokens/s, Running: 0 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.0%, Prefix cache hit rate: 0.0%
(APIServer pid=14743) INFO 05-22 13:46:49 [loggers.py:123] Engine 000: Avg prompt throughput: 0.0 tokens/s, Avg generation throughput: 0.0 tokens/s, Running: 0 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.0%, Prefix cache hit rate: 0.0%


### 使用模型进行多轮会话

**方案一：最简单的写法**

In [23]:
from openai import OpenAI

client = OpenAI(base_url="http://127.0.0.1:8000/v1", api_key="123456")

# messages = [{"role": "system", "content": "你是一个有帮助的助手。"}]  #系统提示词
messages = []

while True:
    user_input = input("我: ")
    if user_input.lower() == 'q': #输入q退出
        break
        
    user_input = user_input + " /no_think"  # 默认思考，加/no_think表示不思考
    messages.append({"role": "user", "content": user_input})
    
    response = client.chat.completions.create(
        model="./Qwen/Qwen3-0.6B",
        messages=messages,
        max_tokens=32768
    )
    
    reply = response.choices[0].message.content
    print(f"Qwen3: {reply}\n")
    
    messages.append({"role": "assistant", "content": reply})

我:  你好，从现在开始你的名字是kokomi，我将以kokomi称呼你


(APIServer pid=15568) INFO 05-22 13:56:25 [chat_utils.py:538] Detected the chat template content format to be 'string'. You can set `--chat-template-content-format` to override this.
(APIServer pid=15568) INFO:     127.0.0.1:33558 - "POST /v1/chat/completions HTTP/1.1" 200 OK
Qwen3: <think>

</think>

你好！我是kokomi，很高兴和你交谈。有什么我可以帮助你的吗？

(APIServer pid=15568) INFO 05-22 13:56:32 [loggers.py:123] Engine 000: Avg prompt throughput: 2.9 tokens/s, Avg generation throughput: 2.2 tokens/s, Running: 0 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.0%, Prefix cache hit rate: 0.0%
(APIServer pid=15568) INFO 05-22 13:56:42 [loggers.py:123] Engine 000: Avg prompt throughput: 0.0 tokens/s, Avg generation throughput: 0.0 tokens/s, Running: 0 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.0%, Prefix cache hit rate: 0.0%


我:  kokomi，介绍一下LLM是什么


(APIServer pid=15568) INFO:     127.0.0.1:59082 - "POST /v1/chat/completions HTTP/1.1" 200 OK
Qwen3: <think>

</think>

你好！我是kokomi，很高兴和你聊天。LLM，即**Large Language Model**，是一种强大的人工智能模型，能够理解和生成自然语言。它能够处理各种文本任务，如写作、翻译、信息查询等。LLM技术已经成为现代人工智能的重要组成部分，广泛应用于多个领域。你对LLM有什么具体的问题或需求吗？

(APIServer pid=15568) INFO 05-22 13:57:02 [loggers.py:123] Engine 000: Avg prompt throughput: 6.7 tokens/s, Avg generation throughput: 7.8 tokens/s, Running: 0 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.0%, Prefix cache hit rate: 16.7%
(APIServer pid=15568) INFO 05-22 13:57:12 [loggers.py:123] Engine 000: Avg prompt throughput: 0.0 tokens/s, Avg generation throughput: 0.0 tokens/s, Running: 0 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.0%, Prefix cache hit rate: 16.7%


我:  0.6B参数的LLM，通常需要多少显存才能部署


(APIServer pid=15568) INFO 05-22 13:57:42 [loggers.py:123] Engine 000: Avg prompt throughput: 17.0 tokens/s, Avg generation throughput: 7.2 tokens/s, Running: 1 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.4%, Prefix cache hit rate: 30.1%
(APIServer pid=15568) INFO:     127.0.0.1:45970 - "POST /v1/chat/completions HTTP/1.1" 200 OK
Qwen3: <think>

</think>

0.6B 的 LLM（如 **Bert** 或 **GPT-3.5**）通常需要 **约 4GB 或 8GB 的显存** 来部署。具体需求会根据 GPU 的型号和优化方式有所不同。如果你有具体的 GPU 型号或使用场景，我可以帮你更精确地估算。

(APIServer pid=15568) INFO 05-22 13:57:52 [loggers.py:123] Engine 000: Avg prompt throughput: 0.0 tokens/s, Avg generation throughput: 0.5 tokens/s, Running: 0 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.0%, Prefix cache hit rate: 30.1%


我:  q


(APIServer pid=15568) INFO 05-22 13:58:02 [loggers.py:123] Engine 000: Avg prompt throughput: 0.0 tokens/s, Avg generation throughput: 0.0 tokens/s, Running: 0 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.0%, Prefix cache hit rate: 30.1%


**方案二：流式输出**

In [27]:
from openai import OpenAI
from IPython.display import display, Markdown

client = OpenAI(base_url="http://127.0.0.1:8000/v1", api_key="abcdefg")
reply_display = None

# messages = [{"role": "system", "content": "你是一个有帮助的助手。"}]
message = []

while True:
    user_input = input("我: ")    # 默认思考
    if user_input.lower() == 'q': # 输入q退出
        break

    # 1. 显示用户输入（一次性，不更新）
    display(Markdown(f"**我:** {user_input}"))
    messages.append({"role": "user", "content": user_input})
    
    # 流式请求
    stream = client.chat.completions.create(
        model="./Qwen/Qwen3-0.6B",
        messages=messages,
        stream=True,
        max_tokens=8196
    )
    
    # Jupyter 流式显示：不断刷新同一个输出区域
    full_response = ""
    for chunk in stream:
        delta = chunk.choices[0].delta
        if delta.content is not None:
            token = delta.content
            # 替换标签为颜色标记
            token = token.replace("<think>", "<span style='color:gray'>[思考] ")
            token = token.replace("</think>", "</span>")
            full_response += token
            if reply_display is None:
                # 第一次显示，创建可更新区域
                reply_display = display(Markdown(f"**Qwen3:** {full_response}"), display_id=True)
            else:
                # 后续直接更新同一个区域
                reply_display.update(Markdown(f"**Qwen3:** {full_response}"))
    
    print()  # 换行
    messages.append({"role": "assistant", "content": full_response})
    # 重置 display 句柄，为下一轮做准备
    reply_display = None

我:  你好，你是谁


**我:** 你好，你是谁

(APIServer pid=15568) INFO:     127.0.0.1:53362 - "POST /v1/chat/completions HTTP/1.1" 200 OK


**Qwen3:** <span style='color:gray'>[思考] 

</span>

你好！我是 kokomi，很高兴和你聊天。我是你的AI助手，可以帮你解答各种问题和提供帮助。如果你有任何问题或需要帮助，随时告诉我！


(APIServer pid=15568) INFO 05-22 14:12:52 [loggers.py:123] Engine 000: Avg prompt throughput: 83.9 tokens/s, Avg generation throughput: 4.0 tokens/s, Running: 0 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.0%, Prefix cache hit rate: 73.9%
(APIServer pid=15568) INFO 05-22 14:13:02 [loggers.py:123] Engine 000: Avg prompt throughput: 0.0 tokens/s, Avg generation throughput: 0.0 tokens/s, Running: 0 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.0%, Prefix cache hit rate: 73.9%


...

**我:** 那VLA呢

(APIServer pid=15568) INFO:     127.0.0.1:38394 - "POST /v1/chat/completions HTTP/1.1" 200 OK


**Qwen3:** <span style='color:gray'>[思考] 
好的，用户之前问了VLM和LLM的区别，现在又问了VLA，我需要确认用户是否在提到VLM的另一个变种，或者可能有拼写错误。首先，VLA通常是指“Very Large Visual Language Model”，也就是VLM，所以用户可能打错了，或者想了解另一个变体。不过根据之前的对话，用户已经明确提到了VLM，所以可能需要进一步解释。另外，用户可能对多模态任务感兴趣，VLA确实是多模态的，所以需要确认这一点。需要确保回答准确，并且解释清楚VLA和VLM的区别，以及其应用场景。同时，保持友好和开放的态度，让用户感到被重视和帮助。
</span>

VLA 是 **Very Large Visual Language Model**（视觉语言模型）的缩写，即 VLM（Very Large Language Model），与 VLM（Very Large Model）不同，它**同时处理文本和图像**的任务。VLA 在视觉和语言任务上表现更优越，适用于图像生成、视频内容创作、多模态问答等场景。

### 与 VLM 的区别：
- **VLM**：仅依赖语言模型（LLM），不涉及视觉信息。
- **VLA**：结合视觉和语言能力，支持多模态任务。

### 应用场景：
- **VLA**：图像生成、视频内容创作、多模态问答等。
- **VLM**：文本生成、翻译、信息查询等。

如果你有具体的应用场景或想了解更详细的信息，我很乐意为你解答！


(APIServer pid=15568) INFO 05-22 14:15:57 [loggers.py:123] Engine 000: Avg prompt throughput: 130.3 tokens/s, Avg generation throughput: 33.0 tokens/s, Running: 0 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.0%, Prefix cache hit rate: 75.6%
(APIServer pid=15568) INFO 05-22 14:16:07 [loggers.py:123] Engine 000: Avg prompt throughput: 0.0 tokens/s, Avg generation throughput: 0.0 tokens/s, Running: 0 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.0%, Prefix cache hit rate: 75.6%


我:  q


### 关闭服务

In [28]:
process.terminate()
process.wait()
print("服务已关闭")

(APIServer pid=15568) WARNING 05-22 14:16:35 [launcher.py:98] port 8000 is used by process psutil.Process(pid=15568, name='vllm', status='running') launched with command:
(APIServer pid=15568) WARNING 05-22 14:16:35 [launcher.py:98] /home/kokomi/anaconda3/envs/mamba/bin/python3.12 /home/kokomi/anaconda3/envs/mamba/bin/vllm serve ./Qwen/Qwen3-0.6B --host 127.0.0.1 --port 8000 --gpu-memory-utilization 0.5
(APIServer pid=15568) INFO 05-22 14:16:35 [launcher.py:101] Shutting down FastAPI HTTP server.


[rank0]:[W522 14:16:36.593515032 ProcessGroupNCCL.cpp:1538] Warning: WARNING: destroy_process_group() was not called before program exit, which can leak resources. For more info, please see https://pytorch.org/docs/stable/distributed.html#shutdown (function operator())
(APIServer pid=15568) INFO:     Shutting down
(APIServer pid=15568) INFO:     Waiting for application shutdown.
(APIServer pid=15568) INFO:     Application shutdown complete.


服务已关闭


## 模型量化

模型量化就是通过降低模型参数的数值精度（如从16位浮点数降到4位整数），以极小的性能损失换取模型体积缩小、显存占用降低和推理速度提升的压缩技术。

动态量化：不需要提前下载量化模型，框架会在加载时自动把 FP16/BF16 的模型压缩成 8-bit或4-bit，极大节省显存。

### 检查bitsandbyte版本

In [47]:
# 检查版本是新的
!pip install -U bitsandbytes

### 8-bit量化

In [12]:
def load_model_8bit(model_name):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    # 8bit量化配置
    bnb_config = BitsAndBytesConfig(load_in_8bit=True)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        #load_in_8bit=True,                  # 直接加这一句就行，已经过时
        quantization_config=bnb_config,      # 新的写法
        device_map="auto"
    )
    return model, tokenizer

model_name = "./Qwen/Qwen3-0.6B"
print_gpu_memory("加载8bit量化模型前")
model, tokenizer = load_model_8bit(model_name)
print_gpu_memory("加载8bit量化模型后")

[加载8bit量化模型前] 已用: 0.01GB | 缓存: 0.15GB | 总计: 6.00GB | 剩余: 5.99GB


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

[加载8bit量化模型后] 已用: 0.74GB | 缓存: 1.20GB | 总计: 6.00GB | 剩余: 5.26GB


In [13]:
# 清理缓存
del model
del tokenizer
CleanMemory()

[缓存已清理] 已用: 0.01GB | 缓存: 0.15GB | 总计: 6.00GB | 剩余: 5.99GB


### 4-bit量化
4-bit压缩非常狠，如果用简单的压缩方法，模型会直接“变傻”，BitsAndBytes 库使用了很多复杂的补救算法。

In [15]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

def load_model_4bit(model_name):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    
    # 1. 定义量化配置
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,                       # 开启 4-bit 量化
        bnb_4bit_compute_dtype=torch.bfloat16,   # 计算时使用的数据类型，Tensor Core不支持 4-bit的数学运算，计算时还原成fp16
        #bnb_4bit_compute_dtype=torch.float16,   # 上面不能用就用下面的
        bnb_4bit_quant_type="nf4",               # 量化类型，nf4 效果最好
        bnb_4bit_use_double_quant=True,          # 使用双量化，再省一点点显存，对缩放因子再做一次量化
    )

    # 2. 加载模型时传入 quantization_config
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,  # 传入量化配置
        device_map="auto"
    )
    return model, tokenizer

In [16]:
model_name = "./Qwen/Qwen3-0.6B"
print_gpu_memory("加载4bit量化模型前")
model, tokenizer = load_model_4bit(model_name)
print_gpu_memory("加载4bit量化模型后")

[加载4bit量化模型前] 已用: 0.01GB | 缓存: 0.15GB | 总计: 6.00GB | 剩余: 5.99GB


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

[加载4bit量化模型后] 已用: 0.51GB | 缓存: 1.03GB | 总计: 6.00GB | 剩余: 5.49GB


In [17]:
# 清理缓存
del model
del tokenizer
CleanMemory()

[缓存已清理] 已用: 0.01GB | 缓存: 0.59GB | 总计: 6.00GB | 剩余: 5.99GB


## Lora微调

### 主流微调方式介绍

**全量微调/全参微调**  
SFT:（Supervised Fine-Tuning 监督微调）是指在大模型预训练完成后，利用人工标注的“指令-回复”成对数据对模型进行进一步训练的过程。  
- 相当于直接加载模型，然后用自己的数据直接训练

**参数高效微调/PEFT(Parameter-Efficient Fine-Tuning)**  
Lora:（Low-Rank Adaptation 低秩微调）是一种参数高效的微调技术，其核心思想是在冻结大模型原有巨大参数矩阵的基础上，额外附加一个低秩的“旁路”矩阵来进行训练。微调时，只更新这个规模极小的旁路参数，而保持原模型参数不变，从而大幅降低显存占用和计算成本。  
- 只训练小矩阵，冻结原本的所有参数

**偏好优化（preference optimization / preference learning）**  
- RLHF 人类反馈强化学习：是一整套用“人类偏好 → 奖励模型 → 强化学习”来对齐模型的框架。
- PPO 近端策略优化：RLHF 里最常用的强化学习算法，用来优化“策略模型”。
- DPO 直接偏好优化：一种新方法，直接用偏好数据优化模型，不再训练奖励模型，也不用强化学习。
- GRPO 群体稳健偏好优化：在 DPO 思路上加了“分组鲁棒”，对不同人群/场景的偏好做更均衡的对齐。
- 偏好对是什么？
```json
"chosen"列:   [{"role": "user", "content": "解释一下黑洞"},  
           {"role": "assistant", "content": "黑洞是时空曲率极大的天体，连光都逃不掉……"}],     
"rejected": [{"role": "user", "content": "解释一下黑洞"},  
           {"role": "assistant", "content": "黑洞就是黑色的洞，很黑很黑……"}] 
"score_chosen"	: 8.0
"score_rejected": 4.0
```
训练时让 chosen 的概率更高，让 rejected 的概率更低

### 安装Lora需要的库

In [18]:
!pip show peft datasets trl
# !pip install peft datasets trl

Name: peft
Version: 0.19.1
Summary: Parameter-Efficient Fine-Tuning (PEFT)
Home-page: https://github.com/huggingface/peft
Author: The HuggingFace team
Author-email: benjamin@huggingface.co
License: Apache
Location: /home/kokomi/anaconda3/envs/mamba/lib/python3.12/site-packages
Requires: accelerate, huggingface_hub, numpy, packaging, psutil, pyyaml, safetensors, torch, tqdm, transformers
Required-by: 
---
Name: datasets
Version: 4.8.5
Summary: HuggingFace community-driven open-source library of datasets
Home-page: https://github.com/huggingface/datasets
Author: HuggingFace Inc.
Author-email: thomas@huggingface.co
License: Apache 2.0
Location: /home/kokomi/anaconda3/envs/mamba/lib/python3.12/site-packages
Requires: dill, filelock, fsspec, httpx, huggingface-hub, multiprocess, numpy, packaging, pandas, pyarrow, pyyaml, requests, tqdm, xxhash
Required-by: trl
---
Name: trl
Version: 1.4.0
Summary: Train transformer language models with reinforcement learning.
Home-page: https://github.com/h

### 配置 LoRA 

In [28]:
from peft import LoraConfig, get_peft_model, TaskType

In [29]:
peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,    # 任务类型：因果语言模型（文本生成）
    r=8,                             # LoRA 的秩（旁路矩阵的维度），常用值 8, 16, 32。越大可学习参数越多，但越费显存
    lora_alpha=32,                   # LoRA 的缩放系数，通常设为 r 的 2-4 倍
    lora_dropout=0.1,                # Dropout 防过拟合
    # target_modules 指定要把 LoRA 旁路加到模型的哪些层。  一般加在比较大的矩阵上，qkvo都属于注意力层，"gate_proj", "up_proj", "down_proj"属于FFN 层
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"] 
)

#加载模型
model_name = "./Qwen/Qwen3-0.6B"
model, tokenizer = load_model(model_name)

# 训练时不需要KV Cach
model.config.use_cache = False
# 启用输入梯度（PEFT高效微调 需要）
model.enable_input_require_grads()

before = sum(p.numel() for p in model.parameters())
print("model params:", before)
# 将 LoRA 挂到 原始模型 上
model = get_peft_model(model, peft_config)
# 打印可训练参数的比例, 也就是lora的参数, 一般只占不到 1%
after = model.print_trainable_parameters() 

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

model params: 596049920
trainable params: 2,293,760 || all params: 598,343,680 || trainable%: 0.3834


In [30]:
# del model
# del tokenizer
# CleanMemory()

### 准备数据

#### 临时使用几条ai生成的数据写入jsonl文件充当数据集

In [68]:
import json
def create_temp_dataset():
    vla_data = [
        {"messages": [{"role": "user", "content": "什么是VLA模型？"}, {"role": "assistant", "content": "VLA是视觉-语言-动作模型（Vision-Language-Action Model）的缩写，它将视觉感知、语言理解和动作执行结合在一起，用于具身智能和机器人控制领域。"}]},
        {"messages": [{"role": "user", "content": "VLA的英文全称是什么？"}, {"role": "assistant", "content": "VLA的英文全称是 Vision-Language-Action Model，即视觉-语言-动作模型。"}]},
        {"messages": [{"role": "user", "content": "VLA模型能做什么？"}, {"role": "assistant", "content": "VLA模型可以让机器人看懂环境、听懂人类指令，并直接输出可执行的物理动作指令，实现从感知到行动的闭环控制。"}]},
        {"messages": [{"role": "user", "content": "VLA和传统大语言模型有什么区别？"}, {"role": "assistant", "content": "传统大语言模型只处理文本，输出也是文本；VLA模型不仅接受文本和图像输入，还能输出物理动作指令，实现了从'说'到'做'的跨越。"}]},
        {"messages": [{"role": "user", "content": "VLA模型的三个核心模块是什么？"}, {"role": "assistant", "content": "VLA模型的三个核心模块是：Vision（视觉感知）、Language（语言理解）和Action（动作执行），分别负责看、听和做。"}]},
        {"messages": [{"role": "user", "content": "RT-2是什么？"}, {"role": "assistant", "content": "RT-2（Robotic Transformer 2）是谷歌提出的经典VLA模型，它将大语言模型的知识与机器人视觉-动作数据结合，展现了强大的泛化能力。"}]},
        {"messages": [{"role": "user", "content": "OpenVLA是什么？"}, {"role": "assistant", "content": "OpenVLA是一个开源的VLA模型，参数量约7B，基于Prismatic视觉编码器和Llama语言模型构建，支持多机器人平台微调部署。"}]},
        {"messages": [{"role": "user", "content": "VLA模型怎么控制机器人？"}, {"role": "assistant", "content": "VLA模型接收摄像头的图像和人类的语言指令，经过视觉编码和语言理解后，直接输出机器人关节的目标动作向量（如7自由度机械臂的关节角度），实现端到端控制。"}]},
        {"messages": [{"role": "user", "content": "VLA对具身智能有什么意义？"}, {"role": "assistant", "content": "VLA打通了感知-认知-行动的闭环，使AI不再局限于屏幕里回答问题，而是能理解现实环境并采取物理行动，是通向通用机器人的关键技术路径。"}]},
        {"messages": [{"role": "user", "content": "VLA模型的输入是什么？"}, {"role": "assistant", "content": "VLA模型的输入通常包括两部分：视觉输入（摄像头图像）和语言输入（自然语言指令），两者共同描述当前环境和任务目标。"}]},
        {"messages": [{"role": "user", "content": "VLA模型的输出是什么？"}, {"role": "assistant", "content": "VLA模型的输出是连续的动作向量，例如机械臂各个关节的目标角度或末端执行器的位姿，可以直接驱动机器人执行物理动作。"}]},
        {"messages": [{"role": "user", "content": "VLA是端到端模型吗？"}, {"role": "assistant", "content": "是的，VLA是端到端模型。它从原始视觉和语言输入直接映射到动作输出，中间不需要人工设计特征提取或规则规划模块。"}]},
        {"messages": [{"role": "user", "content": "训练VLA模型需要什么数据？"}, {"role": "assistant", "content": "训练VLA模型需要三元组数据：(图像, 语言指令, 动作)，即在同一时刻记录机器人看到的画面、收到的指令和实际执行的动作。"}]},
        {"messages": [{"role": "user", "content": "VLA模型能泛化到新物体吗？"}, {"role": "assistant", "content": "是的，VLA模型的一大优势就是泛化能力强。得益于预训练语言模型的知识，它可以理解训练中未见过的新物体和新指令。"}]},
        {"messages": [{"role": "user", "content": "VLA和模仿学习有什么关系？"}, {"role": "assistant", "content": "VLA的训练本质上是一种模仿学习，它通过学习人类专家演示的(图像,指令,动作)数据对，让模型学会在类似场景下做出相同动作。"}]},
        {"messages": [{"role": "user", "content": "VLA模型的视觉编码器通常用什么？"}, {"role": "assistant", "content": "VLA模型通常使用预训练的视觉编码器，如ViT（Vision Transformer）或SigLIP，将图像编码为特征向量后与语言特征融合。"}]},
        {"messages": [{"role": "user", "content": "VLA和RLHF有什么区别？"}, {"role": "assistant", "content": "RLHF是大语言模型的强化学习对齐方法，输出仍是文本；VLA则是直接输出动作的端到端模型，关注物理世界的交互，而非文本偏好优化。"}]},
        {"messages": [{"role": "user", "content": "VLA模型有哪些代表性工作？"}, {"role": "assistant", "content": "代表性工作包括：谷歌的RT-1/RT-2、OpenVLA、Octo、以及基于Diffusion Policy的系列工作等。"}]},
        {"messages": [{"role": "user", "content": "VLA模型能理解中文指令吗？"}, {"role": "assistant", "content": "如果VLA的语言基座模型支持中文（如Qwen-VL），那它就能理解中文指令并执行相应动作。语言理解能力主要取决于基座模型。"}]},
        {"messages": [{"role": "user", "content": "VLA的Action模块输出的是什么格式？"}, {"role": "assistant", "content": "Action模块通常输出连续动作向量，最常见的是7维向量，对应机械臂6个关节角度加1个夹爪开合度，取值范围通常归一化到[-1,1]。"}]},
        {"messages": [{"role": "user", "content": "为什么VLA比传统机器人方法更灵活？"}, {"role": "assistant", "content": "传统机器人方法依赖手工规则和固定程序，换任务就要重写代码；VLA通过语言指令就能切换任务，无需重新编程，极大提升了灵活性。"}]},
        {"messages": [{"role": "user", "content": "VLA模型可以用于自动驾驶吗？"}, {"role": "assistant", "content": "理论上可以，VLA的框架适用于任何需要感知-决策-执行的领域。但目前VLA主要用于机械臂操作，自动驾驶还需要更多安全验证。"}]},
        {"messages": [{"role": "user", "content": "VLA训练最大的挑战是什么？"}, {"role": "assistant", "content": "最大的挑战是高质量动作数据的获取。与文本数据不同，(图像,指令,动作)三元组数据需要真实的机器人操作来采集，成本极高且难以大规模获取。"}]},
        {"messages": [{"role": "user", "content": "VLA和VLM有什么区别？"}, {"role": "assistant", "content": "VLM（视觉语言模型）只输出文本描述；VLA（视觉语言动作模型）在VLM基础上增加了Action头部，能输出物理动作指令，是VLM向具身智能的延伸。"}]},
        {"messages": [{"role": "user", "content": "VLA模型的参数量通常多大？"}, {"role": "assistant", "content": "VLA模型参数量差异很大，小的如RT-1约35M参数，大的如RT-2基于55B的PaLI-X，OpenVLA则为7B参数。"}]},
        {"messages": [{"role": "user", "content": "LoRA能用来微调VLA模型吗？"}, {"role": "assistant", "content": "可以，LoRA可以高效微调VLA模型的语言和动作模块，只需训练极少参数就能让模型适应新任务和新机器人平台。"}]},
        {"messages": [{"role": "user", "content": "VLA模型的推理速度够快吗？"}, {"role": "assistant", "content": "这是VLA的一个挑战。大参数模型推理延迟较高（可能超过100ms），而机器人控制通常需要10-50Hz的频率。模型量化和蒸馏是常用加速手段。"}]},
        {"messages": [{"role": "user", "content": "什么是Embodied AI？和VLA什么关系？"}, {"role": "assistant", "content": "Embodied AI（具身智能）是指能在物理环境中交互的AI系统。VLA是具身智能的核心技术路线之一，负责实现感知到行动的端到端映射。"}]},
        {"messages": [{"role": "user", "content": "VLA怎么处理多步任务？"}, {"role": "assistant", "content": "VLA以自回归方式逐步生成动作，每一步接收新的视觉观察，持续输出动作直到任务完成。也可以结合任务规划器将复杂任务分解为子目标。"}]},
    ]
    
    dataset_path = "vla_dataset.jsonl"
    with open(dataset_path, "w", encoding="utf-8") as f:
        for item in vla_data:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")
            
    print(f"✅ 已生成 {len(vla_data)} 条训练数据")
    return dataset_path

dataset_path = create_temp_dataset()

✅ 已生成 29 条训练数据


用pandas看一下数据

In [77]:
import pandas as pd
pd.set_option('display.max_colwidth', None)
df = pd.read_json(dataset_path, lines=True, encoding='utf-8').head(5)
df.style.set_properties(subset=['messages'], **{'text-align': 'left'})

,messages
0,"[{'role': 'user', 'content': '什么是VLA模型？'}, {'role': 'assistant', 'content': 'VLA是视觉-语言-动作模型（Vision-Language-Action Model）的缩写，它将视觉感知、语言理解和动作执行结合在一起，用于具身智能和机器人控制领域。'}]"
1,"[{'role': 'user', 'content': 'VLA的英文全称是什么？'}, {'role': 'assistant', 'content': 'VLA的英文全称是 Vision-Language-Action Model，即视觉-语言-动作模型。'}]"
2,"[{'role': 'user', 'content': 'VLA模型能做什么？'}, {'role': 'assistant', 'content': 'VLA模型可以让机器人看懂环境、听懂人类指令，并直接输出可执行的物理动作指令，实现从感知到行动的闭环控制。'}]"
3,"[{'role': 'user', 'content': 'VLA和传统大语言模型有什么区别？'}, {'role': 'assistant', 'content': ""传统大语言模型只处理文本，输出也是文本；VLA模型不仅接受文本和图像输入，还能输出物理动作指令，实现了从'说'到'做'的跨越。""}]"
4,"[{'role': 'user', 'content': 'VLA模型的三个核心模块是什么？'}, {'role': 'assistant', 'content': 'VLA模型的三个核心模块是：Vision（视觉感知）、Language（语言理解）和Action（动作执行），分别负责看、听和做。'}]"


#### 加载数据集

In [32]:
from datasets import load_dataset

In [33]:
dataset = load_dataset("json", data_files=dataset_path, split="train")

Generating train split: 0 examples [00:00, ? examples/s]

### 设置Lora训练参数

In [37]:
checkpoint_path = "./qwen3-lora-output"
def get_training_args(checkpoint_path):
    training_args = TrainingArguments(
        output_dir=checkpoint_path,    # 模型保存路径
        num_train_epochs=20,                 # 训练轮数
        per_device_train_batch_size=4,       # 批次大小，显存不够可改小为 2
        gradient_accumulation_steps=4,       # 梯度累积，相当于增大 batch size
        learning_rate=3e-4,                  # LoRA 学习率通常可以设大一点
        logging_steps=4,                     # 每 4 步打印一次日志
        save_steps=100,                      # 每 100 步保存一次 checkpoint
        bf16=True,                           # 使用混合精度训练
        # fp16=True,                         # 上面不能用就用这个
        report_to="none",                    # 不上报日志
        remove_unused_columns=False          # 使用自定义 data_collator 时通常需要设为 False
    )
    return training_args
    
training_args = get_training_args(checkpoint_path)                

### 开始训练

In [38]:
from trl import SFTTrainer

In [39]:
# 使用增强版Trainer： SFTTrainer 并开始训练
trainer = SFTTrainer(
    model = model,
    # peft_config=peft_config,          # 可以使用自动挂载lora
    args = training_args,
    train_dataset=dataset,           # 直接传原始数据集
)

print("开始 LoRA 训练...")
trainer.train()

# 6. 保存 LoRA 权重
# 注意：这里只会保存训练好的lora参数，文件非常小（通常只有几十 MB），而不是整个模型
save_path = "./qwen3-06b-lora"
trainer.model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print("训练完成，LoRA 权重已保存至", save_path)

Tokenizing train dataset:   0%|          | 0/29 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


开始 LoRA 训练...


Step,Training Loss
4,3.604596
8,2.456216
12,1.881527
16,1.526522
20,1.270671
24,1.032525
28,0.818810
32,0.657762
36,0.548196
40,0.491524


训练完成，LoRA 权重已保存至 ./qwen3-06b-lora


In [40]:
del model
del tokenizer
CleanMemory()

[缓存已清理] 已用: 1.15GB | 缓存: 1.54GB | 总计: 6.00GB | 剩余: 4.85GB


### 使用lora

推理的时候，我们需要：先加载原来的基座模型，再把 LoRA 权重“贴”上去。

In [41]:
# 加载模型
model_name = "./Qwen/Qwen3-0.6B"
model, tokenizer = load_model(model_name)

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

查看一下生成的模型文件夹内的文件  
- adapter_model.safetensors：LoRA 权重本体
- adapter_config.json：LoRA 的配置单。 记录了 r=8、lora_alpha=32、target_modules 等信息

In [42]:
!ls qwen3-06b-lora

README.md	     adapter_model.safetensors	tokenizer.json
adapter_config.json  chat_template.jinja	tokenizer_config.json


挂载lora

In [43]:
from peft import PeftModel
# 挂载 LoRA 权重
model = PeftModel.from_pretrained(model, save_path) #save_path就是刚刚训练的模型

使用一个官方提供的方便接口测试

In [44]:
def test_llm(model, tokenizer):
    # 切换到评估模式
    model.eval()
    # 开始提问测试
    prompt = "VLA是什么？能干嘛？"
    # 封装到 messages
    messages = [
        {"role": "user", "content": prompt}
    ]
    
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False  # 是否使用思考模式
    ) 
    # 编码
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
    # 推理
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=32768 #最大上下文长度
    
    )
    # 得到输出并转 ids
    output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 
    try:
        # 找到思考结束的符号的id的位置 151668 (</think>)
        index = len(output_ids) - output_ids[::-1].index(151668)
    except ValueError:
        index = 0
    # 思考内容
    thinking_content = tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip("\n")
    # 输出内容
    content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")
    
    print("thinking content:", thinking_content)
    print("content:", content)

    # 清理缓存
    del generated_ids
    del model_inputs

In [45]:
test_llm(model, tokenizer)

thinking content: 
content: VLA（视觉-语言-动作模型，Vision-Language-Action Model）是将视觉感知、语言理解和物理动作控制结合的新型模型，能实现文本描述到机器人动作的映射。


In [46]:
del model
del tokenizer
CleanMemory()

[缓存已清理] 已用: 1.15GB | 缓存: 1.54GB | 总计: 6.00GB | 剩余: 4.85GB


### 删除本节生成的文件（可跳过）

In [47]:
import shutil
import os

print("数据集保存位置：", dataset_path)
print("模型检查点保存位置：", checkpoint_path)
print("Lora保存位置：", save_path)

# 删除文件夹及其所有内容
os.remove(dataset_path) 
shutil.rmtree(checkpoint_path)
shutil.rmtree(save_path)

数据集保存位置： vla_dataset.jsonl
模型检查点保存位置： ./qwen3-lora-output
Lora保存位置： ./qwen3-06b-lora


In [48]:
if not os.path.exists(dataset_path) and not os.path.exists(checkpoint_path) and not os.path.exists(save_path):
    print("成功删除本节生成的文件！")

成功删除本节生成的文件！


## QLoRA量化微调

- 先量化，再微调 -> QLoRA
- 先微调，再量化 -> 量化部署

**加载4bit量化模型**

In [49]:
# 加载4bit量化基座模型（QLoRA论文写的是4bit量化）
model, tokenizer = load_model_4bit(model_name)
print_gpu_memory("加载4bit量化模型后")

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

[加载4bit量化模型后] 已用: 1.66GB | 缓存: 2.12GB | 总计: 6.00GB | 剩余: 4.34GB


**加载数据集**

In [50]:
dataset_path = create_temp_dataset()

✅ 已生成 29 条训练数据


**开始训练**  
lora配置和训练配置在Lora节定义

In [51]:
checkpoint_path = "./qwen3-qlora-output"
trainer = SFTTrainer(
    model = model,
    peft_config=peft_config,          # 可以使用自动挂载lora
    args = get_training_args(checkpoint_path),
    train_dataset=dataset,           # 直接传原始数据集
)
print("开始 QLoRA 训练...")
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


开始 QLoRA 训练...


Step,Training Loss
4,3.708969
8,2.509360
12,1.917898
16,1.541025
20,1.260303
24,1.011517
28,0.807010
32,0.667542
36,0.566542
40,0.518142


TrainOutput(global_step=40, training_loss=1.4508308708667754, metrics={'train_runtime': 61.8764, 'train_samples_per_second': 9.374, 'train_steps_per_second': 0.646, 'total_flos': 100296041693184.0, 'train_loss': 1.4508308708667754})

**保存lora权重**

In [52]:
save_path = "./qwen3-06b-qlora"
trainer.model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print("训练完成，LoRA 权重已保存至", save_path)

训练完成，LoRA 权重已保存至 ./qwen3-06b-qlora


**重新加载4bit量化模型**

In [55]:
model, tokenizer = load_model_4bit(model_name)

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

**挂载lora到模型上**

In [56]:
model = PeftModel.from_pretrained(model, save_path) 

**测试效果**

In [57]:
test_llm(model, tokenizer)

thinking content: 
content: VLA是开放具身智能（Open-Accelerated Virtual Leader）的中文简称，它结合了视觉、语言和动作感知，旨在帮助人类更高效地完成日常任务。


### 清理缓存和本节生成的文件

In [58]:
del model
del tokenizer
CleanMemory()

[缓存已清理] 已用: 0.54GB | 缓存: 1.10GB | 总计: 6.00GB | 剩余: 5.46GB


In [59]:
import shutil
import os

print("数据集保存位置：", dataset_path)
print("模型检查点保存位置：", checkpoint_path)
print("Lora保存位置：", save_path)

# 删除文件夹及其所有内容
os.remove(dataset_path)
shutil.rmtree(checkpoint_path)
shutil.rmtree(save_path)

数据集保存位置： vla_dataset.jsonl
模型检查点保存位置： ./qwen3-qlora-output
Lora保存位置： ./qwen3-06b-qlora


In [60]:
if not os.path.exists(dataset_path) and not os.path.exists(checkpoint_path) and not os.path.exists(save_path):
    print("成功删除本节生成的文件！")

成功删除本节生成的文件！


### 对比Lora和QLora

核心区别：LoRA 和 QLoRA 的根本差异在于基座模型的精度。**LoRA 是在半精度（16-bit）** 的完整基座模型上挂载低秩适配器进行训练，基座权重冻结但信息无损；而 **QLoRA 是先将基座模型量化压缩到 4-bit（NF4格式）**，再在上面挂载适配器训练 **（适配器本身仍为16-bit）**。简言之，QLoRA = 4-bit 量化基座 + LoRA，它通过牺牲基座模型的部分信息精度，换取了**显存占用的大幅断崖式下降。**

适用场景：**LoRA** 适用于显存充足或**训练小参数模型（如3B以下）的场景**，能提供最稳定、最高保真的微调效果，是追求质量的默认首选。**QLoRA** 则是“**显存不足时**的平民救星”，当你只有消费级显卡（如8G/12G显存）却想微调 7B、14B 甚至更大模型时，QLoRA 能让你勉强跑起来；但由于4-bit量化会损失基座表达力，它对数据量要求更高、调参更玄学，微调效果通常略逊于同等条件下的LoRA。

## 其他微调方式  
后续有时间补充相关原理等

*带instruct的基座模型是通过指令微调过的，能生产符合指令和结构化的回答

**GRPO** trl库官方示例代码

```python
from datasets import load_dataset
from trl import GRPOTrainer
from trl.rewards import accuracy_reward

dataset = load_dataset("trl-lib/DeepMath-103K", split="train")

trainer = GRPOTrainer(
    model="Qwen/Qwen2.5-0.5B-Instruct",
    reward_funcs=accuracy_reward,
    train_dataset=dataset,
)
trainer.train()
model = trainer.model #获取训练好的模型
```

GRPO 核心思想  
- 策略模型（Policy Model）：正在微调的模型（比如加了 LoRA 的 BabyLlama），目标是让它生成更符合人类偏好的文本；
- 参考模型（Reference Model）：原始的预训练模型（深拷贝得到，参数固定），作为 “基准”，避免策略模型偏离基础能力；
- 优势值（Advantage）：衡量每个 token 的 “收益”—— 正数表示该 token 符合优化目标（比如人类喜欢），负数表示不符合（人类不喜欢）。  

GRPO 的核心逻辑
- 对比策略模型和参考模型生成 token 的概率，结合优势值衡量 token 的 “正向 / 负向贡献”，通过损失函数 让策略模型 “放大正向贡献的 token 生成概率，缩小负向贡献的 token 生成概率”，同时通过正则化（裁剪 / KL 散度）避免模型更新失控。

**DPO** trl库官方示例代码

```python
from datasets import load_dataset
from trl import DPOTrainer

dataset = load_dataset("trl-lib/ultrafeedback_binarized", split="train")

trainer = DPOTrainer(
    model="Qwen/Qwen3-0.6B",
    train_dataset=dataset,
)
trainer.train()
model = trainer.model #获取训练好的模型
```

DPO损失计算    
对选中的回答计算 其对数概率 和 参考模型给出的对数概率 的差a    越大越好  
对拒绝的回答计算 其对数概率 和 参考模型给出的对数概率 的差b  
计算偏好差 diff = β * (a-b)  
使用sigmoid(diff) 表示“选中比拒绝更受偏好”的概率  
DPO损失 = -log(sigmoid(diff))   

**RewardTrainer** trl库官方示例代码 → 不能直接用于对话，需配合其他步骤（如PPO），输入自然语言，输出分数  
与GRPO的accuracy_reward的区别是：本模型输出随机分数0~10，而accuracy_reward输出0或1

```python
from trl import RewardTrainer
from datasets import load_dataset

dataset = load_dataset("trl-lib/ultrafeedback_binarized", split="train")

trainer = RewardTrainer(
    model="Qwen/Qwen2.5-0.5B-Instruct",
    train_dataset=dataset,
)
trainer.train()
model = trainer.model #获取训练好的模型
```

## 剪枝

模型剪枝（Model Pruning）是神经网络压缩技术的一种，核心是通过移除模型中“不重要”的参数（如权重、神经元），减少模型大小和计算量，从而加速推理、降低资源消耗（显存、功耗）。

非结构化剪枝（Unstructured Pruning）：  
随机移除个别权重（如将某些权重设为0），模型参数变为稀疏矩阵。  
- 优点：压缩率高（可移除90%以上参数）；
- 缺点：硬件支持差（如GPU/TPU对稀疏矩阵加速有限），需特殊库（如SparseML）支持。

结构化剪枝（Structured Pruning）：  
移除整个结构（如神经元、通道、层），保持模型结构规整。  
- 优点：硬件友好（如移除通道后，卷积层计算量直接减少，GPU加速明显）；
- 缺点：压缩率较低（通常移除30%~50%参数）。

剪枝与量化的区别： 
- 剪枝：减少参数
- 量化：降低精度

### 非结构化L1剪枝 
**注意：模型参数不变，矩阵变稀疏矩阵**    
需要支持稀疏矩阵计算的引擎或硬件，才能发挥作用！

- L0范数：向量中非0的元素的个数
- L1范数：在数学上指绝对值
- L2范数：向量各元素的平方和然后求平方根

导入剪枝库和加载基座模型

In [82]:
import torch
from torch.nn.utils import prune

In [83]:
model_name = "Qwen/Qwen3-0.6B"
model, tokenizer = load_model(model_name)

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

- param model: 要剪枝的模型
- param pruning_amount: 剪枝比例（0~1，如0.2表示移除20%的权重）

In [84]:
def apply_pruning(model, pruning_amount=0.2):
    for name, module in model.named_modules():
        if isinstance(module, torch.nn.Linear):                                  # 只对线性层剪枝（Qwen3的注意力层和MLP层的线性层）
            prune.l1_unstructured(module, name='weight', amount=pruning_amount)  # 移除pruning_amount比例的小权重：绝对值非常接近0的权重
            prune.remove(module, 'weight')                                       # 合并剪枝掩码到权重中（避免保存额外参数）
    return model

In [85]:
before = sum(p.numel() for p in model.parameters())

# 应用剪枝（例如移除20%的权重）
pruned_model = apply_pruning(model, pruning_amount=0.2)

after = sum(p.numel() for p in pruned_model.parameters())
print("model params is same：", before==after)

# 保存剪枝后的模型和tokenizer
pruned_save_path = "./pruned_qwen3_0.6b"
pruned_model.save_pretrained(pruned_save_path)
tokenizer.save_pretrained(pruned_save_path)

model params is same： True


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./pruned_qwen3_0.6b/tokenizer_config.json',
 './pruned_qwen3_0.6b/chat_template.jinja',
 './pruned_qwen3_0.6b/tokenizer.json')

清理缓存和删除生成文件

In [86]:
del model
del tokenizer
del pruned_model
CleanMemory()

[缓存已清理] 已用: 0.54GB | 缓存: 0.73GB | 总计: 6.00GB | 剩余: 5.46GB


In [87]:
import shutil
shutil.rmtree(pruned_save_path)

### 结构化剪枝 
**注意：模型参数减少**  

#### 行剪枝

计算每一行（输出维度）的重要性（L2范数），把不重要的行直接删除，矩阵变小。

In [109]:
import torch
import torch.nn as nn

def structured_prune_linear(layer, prune_ratio=0.2):
    """对线性层进行行级别的结构化剪枝"""
    weight = layer.weight.data                   # 权重矩阵
    num_rows = weight.shape[0]                   # 行数
    num_to_prune = int(num_rows * prune_ratio)   # 需要剪掉的行数
    if num_to_prune == 0:
        return
    
    row_importance = torch.norm(weight, p=2, dim=1)            # 1. 计算每行的重要性 (L2 范数)
    _, sorted_idx = torch.sort(row_importance)                 # 2. 找到最不重要的行的索引
    prune_idx = sorted_idx[:num_to_prune]                      #    取索引中 需要要剪掉的行索引
    keep_idx = sorted_idx[num_to_prune:]                       # 3. 保留其他行的索引 (逻辑删除)
    keep_idx, _ = torch.sort(keep_idx)                         #    恢复行正常的排序
    
    layer.weight = nn.Parameter(weight[keep_idx, :])           # 4. 重构权重矩阵 (参数量变少了)，只取保留的行
    if layer.bias is not None:
        layer.bias = nn.Parameter(layer.bias.data[keep_idx])
        
    layer.out_features = len(keep_idx)                         # 5. 更新层的维度信息

简单测试一下

In [114]:
layer = nn.Linear(2, 4, bias=False) # 4x2张量在计算时会转置成2x4
layer.weight.data = torch.tensor(   # 使用 4x2的张量测试
    [
        [ 0.8, 8.],
        [ 1.,  5.],
        [ 4.,  0.3],
        [ 2.,  1.2]
    ]
)

print("===== 剪枝前 =====")
print(layer.weight.data)

# 执行 2:4 剪枝
structured_prune_linear(layer, prune_ratio=0.25)

print("\n===== 剪枝后 =====")
print(layer.weight.data)

===== 剪枝前 =====
tensor([[0.8000, 8.0000],
        [1.0000, 5.0000],
        [4.0000, 0.3000],
        [2.0000, 1.2000]])

===== 剪枝后 =====
tensor([[0.8000, 8.0000],
        [1.0000, 5.0000],
        [4.0000, 0.3000]])


⚠️ 实际应用须知：在LLM中，如果你删掉了某一层的输出行，必须同时删掉下一层对应的输入列，否则矩阵维度对不上会报错。这也是为什么LLM结构化剪枝工程量大，通常需要写图遍历逻辑来对齐维度。

#### 列剪枝

In [115]:
import torch
import torch.nn as nn

def structured_prune_linear_cols(layer, prune_ratio=0.25):
    """对线性层的输入维度(列)进行结构化剪枝"""
    weight = layer.weight.data
    num_cols = weight.shape[1]                          # 🌟改动1：获取列数 (输入维度)
    num_to_prune = int(num_cols * prune_ratio)
    
    if num_to_prune == 0:
        return  
    
    col_importance = torch.norm(weight, p=2, dim=0)     # 🌟改动2：沿着行(dim=0)计算L2范数，得到每一列的重要性
    _, sorted_idx = torch.sort(col_importance)
    keep_idx = sorted_idx[num_to_prune:]
    keep_idx, _ = torch.sort(keep_idx) 
    layer.weight = nn.Parameter(weight[:, keep_idx])    # 🌟改动3：切片方式反过来，保留所有行，只保留重要的列  
                                                        # 🌟改动4：输入维度剪枝，不需要动 bias
    layer.in_features = len(keep_idx)                   # 🌟改动5：更新的是输入维度

简单测试一下

In [116]:
layer = nn.Linear(4, 2, bias=False) # 2x4张量在计算时会转置成2x4
layer.weight.data = torch.tensor(
    [
        [0.8, 5.0, 4.0, 2.0], 
        [1.0, 6.0, 0.3, 1.2]
    ]
)

print("===== 剪枝前 =====")
print(layer.weight.data)

# 执行 2:4 剪枝
structured_prune_linear_cols(layer, prune_ratio=0.25)

print("\n===== 剪枝后 =====")
print(layer.weight.data)

===== 剪枝前 =====
tensor([[0.8000, 5.0000, 4.0000, 2.0000],
        [1.0000, 6.0000, 0.3000, 1.2000]])

===== 剪枝后 =====
tensor([[5.0000, 4.0000, 2.0000],
        [6.0000, 0.3000, 1.2000]])


### 半结构化2:4剪枝
**注意：模型参数不变，矩阵变稀疏矩阵**  

权重按每4个一组，把绝对值最小的2个设为0。  
满足NVIDIA稀疏张量核心的2:4硬件加速要求。

In [94]:
import torch
import torch.nn as nn

def semi_structured_24_prune_linear(layer):
    """对线性层进行 2:4 半结构化剪枝 (50%稀疏)"""
    weight = layer.weight.data
    original_shape = weight.shape
    
    # 1. 展平权重，按 4 个一组划分 (最后一维必须是4的倍数)
    weight_reshaped = weight.reshape(-1, 4)
    
    # 2. 找到每组中绝对值最小的2个的索引
    # argsort 排序后，前2个就是最小的   对第1维(也就是上一行代码的第二维4)从小到大(默认)排序
    _, indices = torch.sort(weight_reshaped.abs(), dim=1)             # 返回的是数值和索引
    prune_mask = torch.zeros_like(weight_reshaped, dtype=torch.bool)  # 创建全0掩码
    prune_mask.scatter_(1, indices[:, :2], True)                      # 在第1维上，标记前2列为True
    
    # 3. 将标记的位置置为 0，应用剪枝
    weight_reshaped[prune_mask] = 0   
    
    # 4. 恢复原始形状并写回
    layer.weight.data = weight_reshaped.reshape(original_shape)

简单测试一下

In [95]:
layer = nn.Linear(4, 2, bias=False)# 4x2张量在计算时会转置成2x4
layer.weight.data = torch.tensor(  # 使用 2x4x2的张量测试
    [
        [ 0.8, 1.],
        [ 5.,  6.],
        [ 4.,  0.3],
        [ 2.,  1.]
    ]
)

print("===== 剪枝前 =====")
print(layer.weight.data)

# 执行 2:4 剪枝
semi_structured_24_prune_linear(layer)

print("\n===== 剪枝后 =====")
print(layer.weight.data)

===== 剪枝前 =====
tensor([[0.8000, 1.0000],
        [5.0000, 6.0000],
        [4.0000, 0.3000],
        [2.0000, 1.0000]])

===== 剪枝后 =====
tensor([[0., 0.],
        [5., 6.],
        [4., 0.],
        [2., 0.]])
